# Downloading Data
## FloodNet NYC Tutorial
Author: Mark Bauer

# Summary
In this notebook, we'll demonstrate how to download the [FloodNet](https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=FloodNet+NYC&sortBy=relevance&page=1&pageSize=20) data on NYC Open Data using the Socrata API, including downloading the data dictionary and dataset description files.

Please visit the FloodNet [website](https://www.floodnet.nyc/) to learn more.

# Import Libraries

In [1]:
# libraries
import os
import requests
from pathlib import Path
from datetime import datetime
import polars as pl
import geopandas as gpd
import json

In [2]:
# print versions of packages for reproducibility
libraries = {
    "requests": requests.__version__,
    "polars": pl.__version__,
    "geopandas": gpd.__version__,
}

for name, version in libraries.items():
    print(f"{name:10s} {version}")

requests   2.32.5
polars     1.38.1
geopandas  1.1.2


# Introduction

Explore the [FloodNet Catalog](https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=FloodNet+NYC&sortBy=relevance&page=1&pageSize=20) on NYC Open Data, which groups the FloodNet datasets together in one view. From the FloodNet catalog:

> FloodNet's mission is to develop tools for real-time urban flood monitoring, implement these tools to measure flooding in NYC, and make flood data and monitoring tools available in a manner that is accessible and useful to stakeholders inc.: residents, community-based organizations, government agencies, and researchers.

We'll download:
1. FloodNet data dictionary and dataset description files
2. FloodNet [sensor metadata](https://data.cityofnewyork.us/Environment/FloodNet-Sensor-Deployment-Metadata/kb2e-tjy3/about_data)
3. FloodNet [flood events](https://data.cityofnewyork.us/Environment/FloodNet-Street-Flooding-Events-Measured-by-FloodN/aq7i-eu5q/about_data)

# Data Dictionary and Dataset Description

In [3]:
def download_attachments(
    dataset_id,
    folder="data",
    base_url="https://data.cityofnewyork.us"
):
    """Download all attachments for a given NYC Open Data dataset.
    
    Parameters
    ----------
    dataset_id : str
        The NYC Open Data dataset ID.
    folder : str
        Local folder to save attachments to.
    base_url : str
        Base URL for the NYC Open Data API.

    Returns
    -------
    list[str]
        List of saved file paths.
    """

    metadata = requests.get(f"{base_url}/api/views/{dataset_id}.json").json()
    attachments = metadata.get("metadata", {}).get("attachments", [])

    if not attachments:
        print(f"No attachments found for dataset: {dataset_id}")
        return []

    Path(folder).mkdir(parents=True, exist_ok=True)

    saved_files = []
    for attachment in attachments:
        filename = attachment["filename"]
        asset_id = attachment["assetId"]
        url = f"{base_url}/api/views/{dataset_id}/files/{asset_id}?download=true&filename={filename}"

        response = requests.get(url)
        response.raise_for_status()

        filepath = Path(folder) / filename
        with open(filepath, "wb") as f:
            f.write(response.content)

        print(f"Saved: {filepath}")
        saved_files.append(str(filepath))

    return saved_files

Both datasets include the same attachments, so we can use either FloodNet dataset to download attachments.

In [4]:
# sensor metadata dataset id
dataset_id = "kb2e-tjy3" 
download_attachments(dataset_id)

Saved: data/Flood Events_Data Dictionary.xlsx
Saved: data/Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf


['data/Flood Events_Data Dictionary.xlsx',
 'data/Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf']

In [5]:
# sanity check, view attachmentts  in data/ folder
%ls data

Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
manifest.json
sensor-metadata.csv


# Dataset Information and Manifest

In [6]:
def dataset_info(
    dataset_id,
    base_url="https://data.cityofnewyork.us",
    manifest_path="data/manifest.json"
):
    """Print dataset metadata from NYC Open Data and update the manifest file.

    Fetches dataset metadata from the Socrata API, prints a structured summary
    of key fields, and upserts a versioning record into a local manifest JSON file.

    Parameters
    ----------
    dataset_id : str
        The NYC Open Data dataset ID (e.g. 'kb2e-tjy3').
    base_url : str
        Base URL for the NYC Open Data API.
    manifest_path : str
        Path to the local manifest JSON file. Created if it does not exist.
    """

    def parse_timestamp(ts):
        """Convert a Unix timestamp to a human-readable date string."""
        return datetime.fromtimestamp(ts).strftime("%B %d, %Y") if ts else "N/A"

    # fetch dataset metadata from the Socrata API
    metadata = requests.get(f"{base_url}/api/views/{dataset_id}.json").json()

    # extract attachments (e.g. data dictionaries, description PDFs)
    attachments = metadata.get("metadata", {}).get("attachments", [])

    # basic info
    print("=== Dataset Info ===")
    print(f"Name:                {metadata.get('name')}")
    print(f"Description:         {metadata.get('description')}")
    print(f"Date Created:        {parse_timestamp(metadata.get('createdAt'))}")
    print(f"Data Last Updated:   {parse_timestamp(metadata.get('rowsUpdatedAt'))}")
    print(f"Metadata Updated:    {parse_timestamp(metadata.get('metadataUpdatedAt'))}")
    print(f"Views:               {metadata.get('viewCount')}")
    print(f"Downloads:           {metadata.get('downloadCount')}")

    # attribution
    print("\n=== Attribution ===")
    print(f"Data Provided By:    {metadata.get('attribution')}")
    print(f"Source Link:         {metadata.get('attributionLink')}")

    # tags
    print("\n=== Tags ===")
    print(", ".join(metadata.get("tags", [])))

    # license
    print("\n=== License ===")
    print(metadata.get("licenseId", "Unspecified"))

    # attachments (e.g. data dictionary, data description PDF)
    print("\n=== Attachments ===")
    for attachment in attachments:
        print(f"  - {attachment['filename']}")

    # owner
    print("\n=== Owner ===")
    owner = metadata.get("owner", {})
    print(f"Dataset Owner:       {owner.get('displayName')}")

    # publication
    print("\n=== Publication ===")
    print(f"Date Made Public:    {parse_timestamp(metadata.get('publicationDate'))}")
    print(f"Category:            {metadata.get('category')}")
    print(f"Provenance:          {metadata.get('provenance')}")

    # engagement
    print("\n=== Engagement ===")
    print(f"Average Rating:      {metadata.get('averageRating')}")
    print(f"Total Times Rated:   {metadata.get('totalTimesRated')}")
    print(f"Number of Comments:  {metadata.get('numberOfComments')}")

    # column names and data types
    print("\n=== Columns ===")
    for col in metadata.get("columns", []):
        print(f"  - {col.get('fieldName'):30} {col.get('dataTypeName')}")

    # build a slim versioning record from the metadata
    record = {
        "dataset_id": dataset_id,
        "name": metadata.get("name"),
        "data_updated": parse_timestamp(metadata.get("rowsUpdatedAt")),
        "downloaded_at": datetime.now().strftime("%B %d, %Y"),
    }

    # load existing manifest or initialize an empty one
    path = Path(manifest_path)
    manifest = json.loads(path.read_text()) if path.exists() else {"datasets": []}

    # remove any existing record for this dataset before appending the fresh one
    manifest["datasets"] = [d for d in manifest["datasets"] if d["dataset_id"] != dataset_id]
    manifest["datasets"].append(record)

    # write the updated manifest back to disk
    path.write_text(json.dumps(manifest, indent=2))
    print(f"\nManifest updated: {manifest_path}")

In [7]:
# Sensor Deployment Metadata
dataset_id = "kb2e-tjy3"
dataset_info(dataset_id)

=== Dataset Info ===
Name:                FloodNet: Sensor Deployment Metadata
Description:         For the  FloodNet data collection page, please follow <a href="https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=FloodNet+NYC">this link</a>.

This data consists of metadata for each FloodNet sensor including its location. This data can be used to associate FloodNet sensor metadata with Flood Events in the ""FloodNet: Street Flooding Events Measured by FloodNet Sensors"" dataset. The ""Sensor ID"" column in each dataset should be used to make the association.

See attached data description document for more details on data collection, analysis & the publication to cite if using the data or including it in any publication or public presentation.

More information on the FloodNet project can be found here: https://www.floodnet.nyc 

Visualization of the data in an online data dashboard can be found here: https://dataviz.floodnet.nyc
Date Created:        December 18, 2025

In [8]:
# Street Flooding Events Measured by FloodNet Sensors
dataset_id = "aq7i-eu5q"
dataset_info(dataset_id)

=== Dataset Info ===
Name:                FloodNet: Street Flooding Events Measured by FloodNet Sensors
Description:         For the  FloodNet data collection page, please follow <a href="https://data.cityofnewyork.us/browse?Data-Collection_Data-Collection=FloodNet+NYC">this link</a>.

Data presented here were collected by the FloodNet project, which is a collaboration between researchers at academic institutions (New York University and the City University of New York) and NYC government agencies (NYC Department of Environmental Protection, Mayor's Office of Climate & Environmental Justice, Office of Technology & Innovation), working with NYC residents. Data on flood water depth were collected by FloodNet sensors in one minute intervals, then analyzed to develop the flood event summary statistics presented here. A flood event is defined as a series of water depth measurements greater than 10 mm; all measured flood events presented here have undergone QC by a member of the FloodNet tea

In [9]:
# sanity check, Manifest file
manifest_path = "data/manifest.json"

# read in Manifest file
manifest = (
    json.loads(Path(manifest_path)
        .read_text())["datasets"]
)
manifest

[{'dataset_id': 'kb2e-tjy3',
  'name': 'FloodNet: Sensor Deployment Metadata',
  'data_updated': 'March 17, 2026',
  'downloaded_at': 'April 25, 2026'},
 {'dataset_id': 'aq7i-eu5q',
  'name': 'FloodNet: Street Flooding Events Measured by FloodNet Sensors',
  'data_updated': 'February 26, 2026',
  'downloaded_at': 'April 25, 2026'}]

# Dataset Download

In [10]:
def download_data(url: str, dataset_id: str, local_file: str, **params) -> None:
    """
    Download a file from a URL and save it locally.

    Args:
        url (str): The URL to download from
        dataset_id (str): dataset id from NYC Open Data
        local_file (str): The local file path to save the content
        **params: Query parameters (e.g., $limit=500000)
    """
    # ensure the directory exists
    os.makedirs(os.path.dirname(local_file), exist_ok=True)

    # download file
    response = requests.get(url, params=params)
    response.raise_for_status()

    # print the full URL (with query params included)
    print(f"Downloading from: {response.url}\n")

    # save to disk
    with open(local_file, "wb") as f:
        f.write(response.content)

## Metadata Dataset Download

In [11]:
dataset_id = "kb2e-tjy3"
url = f"https://data.cityofnewyork.us/resource/{dataset_id}.csv"
local_file = "data/sensor-metadata.csv"

download_data(url, dataset_id, local_file)

%ls data


Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
manifest.json
sensor-metadata.csv


In [12]:
# sanity check file
local_file = "data/sensor-metadata.csv"
metadata_df = pl.read_csv(local_file, try_parse_dates=True)

# inspect the shape of the data
n_rows, n_columns = metadata_df.shape

print("Shape:")
print(f"Number of rows: {n_rows}")
print(f"Number of columns: {n_columns}")

Shape:
Number of rows: 380
Number of columns: 16


In [13]:
# sanity check, preview data
metadata_df.head()

sensor_name,sensor_id,date_installed,tidally_influenced,date_removed,street_name,borough,zipcode,community_board,council_district,census_tract,nta,latitude,longitude,lowest_point_height_delta_inches,location
str,str,datetime[μs],str,datetime[μs],str,str,i64,i64,str,i64,str,f64,f64,f64,str
"""BK - Georgia Ave/ Livonia Ave""","""BK-livonia-ave-georgia-ave-2tz…",2026-03-06 00:00:00,"""No""",null,"""Georgia Avenue""","""Brooklyn""",11207,305,"""BK05""",3112600,"""BK0503""",40.664423,-73.895839,7.95,"""POINT (-73.895839 40.664423)"""
"""Q - Roosevelt Ave/Prince St""","""Q-roosevelt-ave-prince-st-2t7s…",2026-02-19 00:00:00,"""No""",null,"""Roosevelt Avenue""","""Queens""",11354,407,"""QN07""",4087100,"""QN0707""",40.758898,-73.832241,6.5,"""POINT (-73.832241 40.758898)"""
"""BK - Lott Ave/Thatford Ave""","""BK-lott-ave-thatford-ave-2tzql…",2026-03-06 00:00:00,"""No""",null,"""Lott Avenue""","""Brooklyn""",11212,316,"""BK16""",3092000,"""BK1602""",40.658076,-73.90676,8.19,"""POINT (-73.90676 40.658076)"""
"""BK - Linden Blvd/Vermont St""","""BK-vermont-st-linden-blvd-2tzq…",2026-03-06 00:00:00,"""No""",null,"""Linden Boulevard""","""Brooklyn""",11207,305,"""BK05""",3110400,"""BK0503""",40.659197,-73.888368,11.42,"""POINT (-73.888368 40.659197)"""
"""BK - Halsey St/Saratoga Ave""","""BK-halsey-st-saratoga-ave-2tzm…",2026-03-06 00:00:00,"""No""",null,"""Halsey Street""","""Brooklyn""",11233,303,"""BK03""",3037700,"""BK0302""",40.685532,-73.917807,6.97,"""POINT (-73.917807 40.685532)"""


## Flood Events Dataset Download

In [14]:
dataset_id = "aq7i-eu5q"
url = f"https://data.cityofnewyork.us/resource/{dataset_id}.csv"
local_file = "data/flood-events.csv"
max_limit = 500_000

# assign limit param to a large number to fetch all rows
download_data(url, dataset_id, local_file, **{"$limit": max_limit})

%ls data


Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
manifest.json
sensor-metadata.csv


In [15]:
local_file = "data/flood-events.csv"

events_df = pl.read_csv(
    local_file,
    # increase number of rows to infer schema with columns with sparce data
    infer_schema_length=100_000,
    try_parse_dates=True
)

n_rows, n_columns = events_df.shape

# sanity checks
assert n_rows > 0, "No data loaded — check file!"
assert n_rows != max_limit, (
    f"Row count hit the limit ({max_limit:,}) — data may be truncated!"
)

# inspect shape
print("Shape:")
print(f"Number of rows: {n_rows:,}")
print(f"Number of columns: {n_columns}")

Shape:
Number of rows: 1,887
Number of columns: 13


In [16]:
# sanity check, preview data
events_df.head()

sensor_name,sensor_id,flood_start_time,flood_end_time,max_depth_inches,onset_time_mins,drain_time_mins,duration_mins,duration_above_4_inches_mins,duration_above_12_inches_mins,duration_above_24_inches_mins,flood_profile_depth_inches,flood_profile_time_secs
str,str,datetime[μs],datetime[μs],f64,f64,f64,f64,f64,f64,f64,str,str
"""Q - Beach 84 St""","""Q-beach-84-st-0me680""",2023-10-30 12:00:39,2023-10-30 16:38:19,17.72,133.86,143.81,277.66,211.41,103.6,0.0,"""[0.00, 0.87, 1.02, 1.38, 1.93,…","""[0, 1008, 1134, 1512, 1826, 20…"
"""BX - 1st St/Avenue A""","""BX-1st-st-avenue-a-1zby90""",2025-04-26 23:17:20,2025-04-27 11:55:34,6.18,27.95,730.29,758.24,132.81,0.0,0.0,"""[0.00, 0.71, 1.38, 1.38, 1.38,…","""[0, 64, 129, 132, 196, 260, 32…"
"""Q - Beach 35 St/Beach Channel …","""Q-beach-35-st-beach-channel-dr…",2024-02-09 11:41:01,2024-02-09 13:20:57,5.31,54.51,45.43,99.94,45.09,0.0,0.0,"""[0.00, 0.43, 0.63, 0.71, 0.94,…","""[0, 68, 136, 204, 273, 341, 40…"
"""BX - Tibbett Ave/W 234th St""","""BX-w-234th-st-tibbett-ave-2cak…",2025-08-20 21:05:19,2025-08-21 00:46:19,1.18,80.0,141.0,221.0,0.0,0.0,0.0,"""[0.00, 0.43, 0.47, 0.47, 0.51,…","""[0, 60, 120, 180, 240, 300, 36…"
"""Q - Davenport Ct 1""","""Q-davenport-ct-1-07zks0""",2024-01-10 00:39:25,2024-01-10 02:30:40,2.09,59.82,51.43,111.25,0.0,0.0,0.0,"""[0.00, 0.47, 0.47, 0.43, 0.43,…","""[0, 63, 127, 189, 252, 315, 37…"


# NYC Borough Boundaries
We'll download NYC's borough boundaries GeoJSON file for mapping. Note the geometry column for mapping.

In [17]:
# read in NYC Boroughs geometries from NYC Open Data
path = "https://data.cityofnewyork.us/resource/gthc-hcne.geojson"
boro_gdf = gpd.read_file(path)

# preview data, confirm CRS
print(boro_gdf.crs)
boro_gdf.head()

EPSG:4326


,borocode,boroname,shape_area,shape_leng,geometry
0,5,Staten Island,1623618358.46,325912.288988,"MULTIPOLYGON (((-74.05051 40.56642, -74.05047 ..."
1,1,Manhattan,636631650.451,359537.866079,"MULTIPOLYGON (((-74.01093 40.68449, -74.01193 ..."
2,2,Bronx,1187199300.36,463147.071867,"MULTIPOLYGON (((-73.89681 40.79581, -73.89694 ..."
3,3,Brooklyn,1934462607.43,726953.044632,"MULTIPOLYGON (((-73.86327 40.58388, -73.86381 ..."
4,4,Queens,3041419178.99,887905.076018,"MULTIPOLYGON (((-73.82645 40.59053, -73.82642 ..."


In [18]:
# save locally
boro_gdf.to_parquet('data/boroughs.parquet')

In [19]:
# sanity check, list files
%ls data/

Flood Events_Data Dictionary.xlsx
Floodnet_Open_Data_-_Data_Description_03.05.2026.pdf
boroughs.parquet
flood-events.csv
manifest.json
sensor-metadata.csv
